# Windows Security / Sysmon Anomaly-Detection Pipeline — Corrected & Verified

This notebook rebuilds the pipeline end-to-end:

`JSONL logs -> event filtering -> field normalization -> user/logon/process correlation -> 10-minute user windows -> 69 security features -> per-user/per-hour Median+MAD baseline -> anomaly comparison (rarity + magnitude + severity composite score) -> test any new JSONL file against the baseline`

All outputs below are **recomputed directly from `data.jsonl`** rather than trusted from the original notebook. See the accompanying verification report for a list of every bug found and fixed.

**Set `FILE` below to the path of your training JSONL log file.** A separate `TEST_FILE` cell at the end lets you score any new/suspicious log file against the baseline built here.


In [16]:
import json, math, ipaddress, os
from collections import Counter, defaultdict
from datetime import datetime, timezone
from statistics import mean, pstdev, median

FILE = "normal.jsonl"      # <-- path to your TRAINING JSONL log file
WINDOW_SEC = 600          # 10-minute windows


## Stage 0 — Load raw JSONL and inventory the fields/event mix

In [17]:
# =====================================================================
# STAGE 0 — LOAD
# =====================================================================
raw_records = [json.loads(line) for line in open(FILE, encoding="utf-8")]
print(f"[STAGE 0] loaded {len(raw_records)} raw JSONL records from {FILE}")

keys = set()
for r in raw_records:
    keys.update(r.keys())
mix = Counter((r.get("channel"), r.get("event_id")) for r in raw_records)
print(f"[STAGE 0] {len(keys)} distinct top-level fields present across all records")
print("[STAGE 0] (channel, event_id) counts:")
for (ch, eid), n in sorted(mix.items(), key=lambda kv: str(kv[0])):
    print(f"    {ch:38} {eid:>5} : {n}")

[STAGE 0] loaded 1096 raw JSONL records from normal.jsonl
[STAGE 0] 75 distinct top-level fields present across all records
[STAGE 0] (channel, event_id) counts:
    Microsoft-Windows-Sysmon/Operational       1 : 200
    Microsoft-Windows-Sysmon/Operational      11 : 84
    Microsoft-Windows-Sysmon/Operational      22 : 122
    Microsoft-Windows-Sysmon/Operational       3 : 320
    Microsoft-Windows-Sysmon/Operational       5 : 40
    Security                                4624 : 40
    Security                                4625 : 3
    Security                                4634 : 40
    Security                                4647 : 40
    Security                                4698 : 1
    Security                                4724 : 1
    Security                                4738 : 1
    Security                                4768 : 40
    Security                                4769 : 40
    Security                                5140 : 62
    Security                 

## Stage 1 — Event filtering (keep only security-relevant event IDs)

In [18]:
# =====================================================================
# STAGE 1 — EVENT FILTERING
# =====================================================================
# KEEP_EVENTS is intentionally broader than what appears in this particular
# JSONL so the pipeline is forward-compatible with fuller log captures
# (e.g. LSASS access #10, registry #12/13, image load #7, pipe #17/18,
# file-delete #23, driver #24 are not present in THIS file but are kept
# so the code does not silently break when they show up).
KEEP_EVENTS = [
    # Security
    4624, 4625, 4634, 4647, 4688, 4768, 4769, 4771, 4776,
    4698, 4724, 4738, 5140, 5145,
    # Sysmon
    1, 3, 5, 7, 10, 11, 12, 13, 15, 17, 18, 22, 23, 24,
]

records = [r for r in raw_records if r.get("event_id") in KEEP_EVENTS]
dropped = [r for r in raw_records if r.get("event_id") not in KEEP_EVENTS]
print(f"\n[STAGE 1] kept {len(records)} / dropped {len(dropped)} records "
      f"(KEEP_EVENTS has {len(KEEP_EVENTS)} ids)")
if dropped:
    print("[STAGE 1] dropped breakdown:",
          Counter((r.get("channel"), r.get("event_id")) for r in dropped))

present_ids = {eid for (_, eid) in mix}
missing_from_data = sorted(set(KEEP_EVENTS) - present_ids)
print(f"[STAGE 1] KEEP_EVENTS ids with ZERO occurrences in this file "
      f"(features for these will legitimately read 0 unless injected): {missing_from_data}")


[STAGE 1] kept 1096 / dropped 0 records (KEEP_EVENTS has 28 ids)
[STAGE 1] KEEP_EVENTS ids with ZERO occurrences in this file (features for these will legitimately read 0 unless injected): [7, 10, 12, 13, 15, 17, 18, 23, 24, 4688, 4771, 4776]


## Stage 2 — Field normalization into one clean schema

In [19]:
# =====================================================================
# STAGE 2 — FIELD NORMALIZATION
# =====================================================================
def strip_domain(u):
    """CORP\\alice -> alice. Leaves bare names untouched."""
    return u.split("\\")[-1] if u and "\\" in u else u

def normalize(r):
    eid = r["event_id"]
    raw_user = (r.get("User") or r.get("TargetUserName")
                or r.get("SubjectUserName") or r.get("user"))
    return {
        "host": r.get("computer"), "time": r.get("timestamp"), "event_id": eid,
        "user": strip_domain(raw_user),
        "sid": r.get("TargetUserSid") or r.get("SubjectUserSid"),
        # FIX: LogonId lookup order now also covers the field actually used
        # by 4634 (TargetLogonId) and 4647 (SubjectLogonId) — unchanged from
        # original, verified correct against the data (100% match rate).
        "logon_id": r.get("LogonId") or r.get("TargetLogonId") or r.get("SubjectLogonId"),
        "process_guid": r.get("ProcessGuid"),
        "parent_process_guid": r.get("ParentProcessGuid"),
        "image": r.get("Image"), "cmdline": r.get("CommandLine"),
        "parent_image": r.get("ParentImage"), "parent_user": r.get("ParentUser"),
        "dst_ip": r.get("DestinationIp"), "dst_port": r.get("DestinationPort"),
        "src_ip": r.get("SourceIp") or r.get("IpAddress"), "initiated": r.get("Initiated"),
        "workstation": r.get("WorkstationName") or r.get("Workstation"),
        "logon_type": r.get("LogonType"), "target_user": r.get("TargetUserName"),
        "query_name": r.get("QueryName"), "target_filename": r.get("TargetFilename"),
        "target_object": r.get("TargetObject"), "target_image": r.get("TargetImage"),
        "target_process_guid": r.get("TargetProcessGUID"), "granted_access": r.get("GrantedAccess"),
        "pipe_name": r.get("PipeName"), "image_loaded": r.get("ImageLoaded"),
        "service_name": r.get("ServiceName"), "share_name": r.get("ShareName"),
        "relative_target": r.get("RelativeTargetName"), "status": r.get("Status"),
    }

records = [normalize(r) for r in records]
print(f"\n[STAGE 2] normalized {len(records)} records")
for eid in (4624, 1, 3, 22):
    ex = next((r for r in records if r["event_id"] == eid), None)
    if ex:
        present = {k: v for k, v in ex.items() if v is not None}
        print(f"    event {eid}: {len(present)} populated fields -> {list(present)}")


[STAGE 2] normalized 1096 records
    event 4624: 10 populated fields -> ['host', 'time', 'event_id', 'user', 'sid', 'logon_id', 'src_ip', 'workstation', 'logon_type', 'target_user']
    event 1: 9 populated fields -> ['host', 'time', 'event_id', 'user', 'logon_id', 'process_guid', 'image', 'cmdline', 'parent_user']
    event 3: 10 populated fields -> ['host', 'time', 'event_id', 'user', 'process_guid', 'image', 'dst_ip', 'dst_port', 'src_ip', 'initiated']
    event 22: 6 populated fields -> ['host', 'time', 'event_id', 'user', 'image', 'query_name']


## Stage 3 — User / logon-session / process correlation

In [20]:
# =====================================================================
# STAGE 3 — USER / LOGON-SESSION / PROCESS CORRELATION
# =====================================================================
# Well-known non-interactive / service identities. Their process activity
# is real, but cannot be attributed to a human user via a 4624 interactive
# logon (there usually isn't one for LogonId 0x3e7 == the SYSTEM session).
# We resolve them explicitly to None ("UNATTRIBUTED") rather than silently
# letting the literal string "SYSTEM" become a pseudo-"user" that would
# otherwise pollute the per-user behavioral baselines built in Stage 5.
SYSTEM_ACCOUNTS = {"SYSTEM", "LOCAL SERVICE", "NETWORK SERVICE", "ANONYMOUS LOGON"}

logon_map = {}   # (host, logon_id) -> session info, from Security 4624
proc_map = {}    # process_guid     -> process info, from Sysmon 1

for r in records:
    if r["event_id"] == 4624:
        logon_map[(r["host"], r["logon_id"])] = {
            "user": r["user"], "sid": r["sid"], "src_ip": r["src_ip"],
            "workstation": r["workstation"], "logon_type": r["logon_type"], "time": r["time"],
        }
    elif r["event_id"] == 1:
        proc_map[r["process_guid"]] = {
            "user": r["user"], "logon_id": r["logon_id"], "image": r["image"], "time": r["time"],
        }

print(f"\n[STAGE 3] logon_map entries: {len(logon_map)} | proc_map entries: {len(proc_map)}")

def session_of(r):
    """Find the 4624 session that owns this event's process (if any)."""
    p = proc_map.get(r["process_guid"])
    lid = r["logon_id"] or (p or {}).get("logon_id")
    return logon_map.get((r["host"], lid))

def resolve_user(r):
    # 1) direct identity, unless it's a non-interactive system account
    if r["user"] and r["user"].upper() not in SYSTEM_ACCOUNTS:
        return r["user"], r["sid"]
    # 2) chain: process -> its logon session -> the human account that logged on
    sess = session_of(r)
    if sess and sess["user"] and sess["user"].upper() not in SYSTEM_ACCOUNTS:
        return sess["user"], sess["sid"]
    return None, None

for r in records:
    r["resolved_user"], r["resolved_sid"] = resolve_user(r)
    sess = session_of(r)
    if sess:
        r["login_ip"] = sess["src_ip"]
        r["login_type"] = sess["logon_type"]
        r["login_workstation"] = sess["workstation"]
    else:
        r["login_ip"] = r["login_type"] = r["login_workstation"] = None

n_unattributed = sum(1 for r in records if r["resolved_user"] is None)
n_system = sum(1 for r in records if (r["user"] or "").upper() in SYSTEM_ACCOUNTS)
print(f"[STAGE 3] resolved_user is None (unattributed, e.g. SYSTEM svc procs): {n_unattributed}")
print(f"[STAGE 3]   of which raw user was a system/service account        : {n_system}")
print(f"[STAGE 3] events carrying login_ip enrichment                     : "
      f"{sum(1 for r in records if r['login_ip'])}")
print("[STAGE 3] resolved user population:",
      Counter(r["resolved_user"] for r in records if r["resolved_user"]))


[STAGE 3] logon_map entries: 40 | proc_map entries: 200
[STAGE 3] resolved_user is None (unattributed, e.g. SYSTEM svc procs): 80
[STAGE 3]   of which raw user was a system/service account        : 80
[STAGE 3] events carrying login_ip enrichment                     : 746
[STAGE 3] resolved user population: Counter({'alice': 259, 'bob': 256, 'charlie': 254, 'diana': 247})


## Stage 4 — 10-minute user windows

In [21]:
# =====================================================================
# STAGE 4 — 10-MINUTE USER WINDOWS
# =====================================================================
def slot_of(ts):
    dt = datetime.fromisoformat(ts.replace("Z", "+00:00"))
    return int(dt.timestamp()) // WINDOW_SEC * WINDOW_SEC

def slot_label(s):
    return datetime.fromtimestamp(s, tz=timezone.utc).strftime("%m-%d %H:%M")

def day_of(s):
    return datetime.fromtimestamp(s, tz=timezone.utc).strftime("%Y-%m-%d")

def hour_of(s):
    return datetime.fromtimestamp(s, tz=timezone.utc).hour

# FIX: bucket by the CORRELATED resolved_user everywhere (previously the
# feature-engineering / baseline stages of the notebook re-bucketed on the
# raw "user" field, silently discarding all the Stage-3 correlation work
# and turning "SYSTEM" into its own pseudo-user).
buckets = defaultdict(list)
for r in records:
    buckets[(r["resolved_user"] or "UNATTRIBUTED", slot_of(r["time"]))].append(r)

real_users = sorted({u for (u, _s) in buckets if u != "UNATTRIBUTED"})
print(f"\n[STAGE 4] distinct (user, 10-min-slot) buckets: {len(buckets)}")
print(f"[STAGE 4] real users found: {real_users}")
print(f"[STAGE 4] UNATTRIBUTED-bucket events (excluded from per-user baselines): "
      f"{sum(len(v) for (u, s), v in buckets.items() if u == 'UNATTRIBUTED')}")


[STAGE 4] distinct (user, 10-min-slot) buckets: 653
[STAGE 4] real users found: ['alice', 'bob', 'charlie', 'diana']
[STAGE 4] UNATTRIBUTED-bucket events (excluded from per-user baselines): 80


## Stage 5 — Security helpers (LOLBins, suspicious pairs, entropy, private-IP check)

In [22]:
# =====================================================================
# STAGE 5 — SECURITY HELPERS (LOLBins, suspicious pairs, entropy, etc.)
# =====================================================================
LOLBINS = {"certutil.exe", "rundll32.exe", "powershell.exe", "powershell_ise.exe", "mshta.exe",
           "wscript.exe", "cscript.exe", "regsvr32.exe", "cmd.exe", "wmic.exe", "bitsadmin.exe",
           "schtasks.exe", "reg.exe", "whoami.exe", "net.exe", "net1.exe", "nltest.exe",
           "msbuild.exe", "csc.exe", "installutil.exe", "regasm.exe", "cmstp.exe", "bash.exe", "psexec.exe"}

SHELLS = {"cmd.exe", "powershell.exe", "powershell_ise.exe", "wscript.exe", "cscript.exe", "bash.exe", "mshta.exe"}

# Expanded modestly vs. the original set (documented assumption — see report):
# added the reverse cmd<->powershell pairs and a couple more common
# office/browser -> LOLBin spawns that are standard T1059/T1204 patterns.
SUSPICIOUS_PAIRS = {
    ("winword.exe", "powershell.exe"), ("winword.exe", "cmd.exe"), ("winword.exe", "mshta.exe"),
    ("excel.exe", "powershell.exe"), ("excel.exe", "cmd.exe"),
    ("outlook.exe", "powershell.exe"), ("outlook.exe", "cmd.exe"), ("outlook.exe", "mshta.exe"),
    ("msedge.exe", "powershell.exe"), ("msedge.exe", "cmd.exe"),
    ("chrome.exe", "powershell.exe"), ("chrome.exe", "cmd.exe"),
    ("wscript.exe", "powershell.exe"), ("mshta.exe", "cmd.exe"), ("mshta.exe", "powershell.exe"),
    ("explorer.exe", "mshta.exe"),
    ("cmd.exe", "powershell.exe"), ("powershell.exe", "cmd.exe"),
}

def base_name(path):
    return (path or "").lower().split("\\")[-1]

def is_lolbin(image):
    return base_name(image) in LOLBINS

def lolbin_path_anomaly(image):
    if not is_lolbin(image):
        return False
    p = (image or "").lower()
    return "system32" not in p and "syswow64" not in p

def suspicious_parent_child(parent_image, image):
    return (base_name(parent_image), base_name(image)) in SUSPICIOUS_PAIRS

def parent_user_mismatch(parent_user, user):
    pu, u = parent_user or "", user or ""
    return bool(pu and u and strip_domain(pu) != strip_domain(u))

def special_char_density(s):
    if not s:
        return 0.0
    return sum(1 for c in s if not c.isalnum()) / len(s)

def shannon_entropy(s):
    if not s:
        return 0.0
    cnt = Counter(s)
    n = len(s)
    return -sum((c / n) * math.log2(c / n) for c in cnt.values())

# FIX: the original is_external() only matched "172.16." (a single /24),
# missing the rest of the RFC1918 172.16.0.0/12 block (172.17.x-172.31.x),
# which would have misclassified those addresses as "external". Replaced
# with a proper ipaddress-based private-network check.
_PRIVATE_NETS = [ipaddress.ip_network(n) for n in
                  ("10.0.0.0/8", "172.16.0.0/12", "192.168.0.0/16", "127.0.0.0/8")]

def is_external(ip):
    if not ip:
        return False
    try:
        addr = ipaddress.ip_address(ip)
    except ValueError:
        return False
    return not any(addr in net for net in _PRIVATE_NETS)

def has_encoded(cmdline):
    if not cmdline:
        return False
    ls = cmdline.lower()
    return any(k in ls for k in ("-enc", "-e ", "frombase64string", " -w hidden", "-windowstyle hidden", "bypass"))

print(f"\n[STAGE 5] helpers ready — {len(LOLBINS)} LOLBins, {len(SUSPICIOUS_PAIRS)} suspicious parent/child pairs")


[STAGE 5] helpers ready — 24 LOLBins, 18 suspicious parent/child pairs


## Stage 6 — Global process timeline (lifetime, connections, beaconing)

In [23]:
# =====================================================================
# STAGE 6 — GLOBAL PROCESS TIMELINE (lifetime, beaconing, connections)
# =====================================================================
def epoch(ts):
    return int(datetime.fromisoformat(ts.replace("Z", "+00:00")).timestamp())

proc_start, proc_end = {}, {}
proc_image, proc_cmdline = {}, {}
proc_parent, proc_parent_image, proc_parent_user, proc_user = {}, {}, {}, {}
conns_by_proc = defaultdict(list)          # process_guid -> [(epoch, dst_ip, dst_port)]
downloads_by_proc = Counter()

for r in records:
    pg = r.get("process_guid")
    if not pg:
        continue
    if r["event_id"] == 1:
        proc_start[pg] = epoch(r["time"]); proc_image[pg] = r.get("image")
        proc_cmdline[pg] = r.get("cmdline"); proc_parent[pg] = r.get("parent_process_guid")
        proc_parent_image[pg] = r.get("parent_image"); proc_parent_user[pg] = r.get("parent_user")
        proc_user[pg] = r.get("user")
    elif r["event_id"] == 5:
        proc_end[pg] = epoch(r["time"])
    elif r["event_id"] == 3:
        conns_by_proc[pg].append((epoch(r["time"]), r.get("dst_ip"), r.get("dst_port")))
    elif r["event_id"] == 15:
        downloads_by_proc[pg] += 1

def beacon_regularity(pg):
    """1.0 = perfectly regular beacon-like interval. None if too few connections."""
    by_ip = defaultdict(list)
    for t, ip, _port in conns_by_proc[pg]:
        by_ip[ip].append(t)
    best = None
    for _ip, ts in by_ip.items():
        if len(ts) < 4:
            continue
        ts_sorted = sorted(ts)
        gaps = [b - a for a, b in zip(ts_sorted, ts_sorted[1:])]
        if mean(gaps) <= 0:
            continue
        reg = 1.0 - min(1.0, pstdev(gaps) / mean(gaps))
        best = reg if best is None else max(best, reg)
    return best

proc_feats = {}
for pg in proc_start:
    lifetime = proc_end.get(pg, proc_start[pg]) - proc_start[pg]
    img = proc_image.get(pg) or ""
    cmdl = proc_cmdline.get(pg) or ""
    parent_pg = proc_parent.get(pg)
    proc_feats[pg] = {
        "unique_ips": len({c[1] for c in conns_by_proc[pg]}),
        "downloads": downloads_by_proc.get(pg, 0),
        "cmd_char_length": len(cmdl),
        "special_char_density": special_char_density(cmdl),
        "is_lolbin": is_lolbin(img),
        "lolbin_path_anomaly": lolbin_path_anomaly(img),
        "suspicious_parent_child": suspicious_parent_child(proc_parent_image.get(pg), img),
        "parent_user_mismatch": parent_user_mismatch(proc_parent_user.get(pg), proc_user.get(pg)),
        "lifetime_seconds": lifetime,
        "is_micro_duration": lifetime < 5,
        "parent_dead_before_spawn": bool(parent_pg and parent_pg in proc_end
                                          and proc_end[parent_pg] < proc_start[pg]),
        "beacon_regularity": beacon_regularity(pg),
    }

print(f"[STAGE 6] processes tracked: {len(proc_start)} | with an Event-5 end: {len(proc_end)}")

[STAGE 6] processes tracked: 200 | with an Event-5 end: 40


## Stage 7 — Behavioral novelty (time-ordered, leak-free "first seen")

In [24]:
# =====================================================================
# STAGE 7 — BEHAVIORAL NOVELTY (TIME-ORDERED, LEAK-FREE "FIRST SEEN")
# =====================================================================
# FIX (critical): the original notebook built ONE global "seen" set from
# the ENTIRE dataset (all 10 days, including the window under test) before
# computing "new_*" novelty features. Because every value observed inside
# any given window is, trivially, a member of the very set built from that
# same window, new_dst_ips / new_domains / new_images / new_cmdlines /
# new_workstations / unusual_port_for_process could NEVER fire on real
# data — they were structurally always 0. The only reason the notebook's
# injected-attack demo "worked" is that the injected records were kept
# out of that global set entirely (a special case, not a general fix).
#
# The corrected approach processes each user's events in time order and
# marks a value "new" only if it was never seen for that user BEFORE the
# current event — a proper incremental novelty/first-seen check that does
# not leak future or same-window information.
seen_ip, seen_domain, seen_image, seen_cmdline, seen_ws, seen_imgport = (
    defaultdict(set) for _ in range(6)
)
is_new = {}  # id(record) -> dict of novelty booleans, keyed by record identity

for u in real_users:
    user_records = sorted(
        (r for r in records if r["resolved_user"] == u),
        key=lambda r: r["time"],
    )
    for r in user_records:
        flags = {}
        if r.get("dst_ip"):
            flags["new_ip"] = r["dst_ip"] not in seen_ip[u]
            seen_ip[u].add(r["dst_ip"])
        if r.get("query_name"):
            flags["new_domain"] = r["query_name"] not in seen_domain[u]
            seen_domain[u].add(r["query_name"])
        if r["event_id"] == 1 and r.get("image"):
            flags["new_image"] = r["image"] not in seen_image[u]
            seen_image[u].add(r["image"])
        if r["event_id"] == 1 and r.get("cmdline"):
            flags["new_cmdline"] = r["cmdline"] not in seen_cmdline[u]
            seen_cmdline[u].add(r["cmdline"])
        if r["event_id"] == 4624 and r.get("workstation"):
            flags["new_ws"] = r["workstation"] not in seen_ws[u]
            seen_ws[u].add(r["workstation"])
        if r.get("dst_port") and r.get("image"):
            key = (base_name(r["image"]), r["dst_port"])
            flags["new_imgport"] = key not in seen_imgport[u]
            seen_imgport[u].add(key)
        is_new[id(r)] = flags

n_new_ip = sum(1 for r in records if is_new.get(id(r), {}).get("new_ip"))
n_total_ip_events = sum(1 for r in records if r.get("dst_ip"))
print(f"\n[STAGE 7] first-time destination IPs (time-ordered, per user): "
      f"{n_new_ip} / {n_total_ip_events} connection events")
print("[STAGE 7] (expected: only the FIRST time each user's box talks to a given "
      "IP/domain/image/cmdline/workstation/port combo counts as novel)")


[STAGE 7] first-time destination IPs (time-ordered, per user): 16 / 320 connection events
[STAGE 7] (expected: only the FIRST time each user's box talks to a given IP/domain/image/cmdline/workstation/port combo counts as novel)


## Stage 8 — Per-window feature engineering (69 features)

In [25]:
# =====================================================================
# STAGE 8 — PER-WINDOW FEATURE ENGINEERING (69 features)
# =====================================================================
def count_id(rs, eid):
    return sum(1 for r in rs if r["event_id"] == eid)

def nunique(rs, eid, field):
    return len({r[field] for r in rs if r["event_id"] == eid and r.get(field) is not None})

def window_features(rs):
    n_4625 = count_id(rs, 4625); n_4624 = count_id(rs, 4624)
    conns = [r for r in rs if r["event_id"] == 3]
    ext = [r for r in conns if is_external(r.get("dst_ip"))]
    inbound = [r for r in conns if r.get("initiated") in (False, "false", "False")]
    pgs = {r.get("process_guid") for r in rs if r.get("process_guid")}
    pf = [proc_feats[p] for p in pgs if p in proc_feats]

    f = {}
    # --- B1: event-volume counters, one per event ID we track ---
    for eid in KEEP_EVENTS:
        f[f"n_{eid}"] = count_id(rs, eid)

    # --- B2: cardinality (how many DISTINCT values, not just count) ---
    f["nuniq_3_DestinationIp"] = nunique(rs, 3, "dst_ip")
    f["nuniq_3_DestinationPort"] = nunique(rs, 3, "dst_port")
    f["nuniq_1_Image"] = nunique(rs, 1, "image")
    f["nuniq_22_QueryName"] = nunique(rs, 22, "query_name")
    f["nuniq_11_TargetFilename"] = nunique(rs, 11, "target_filename")
    f["nuniq_4624_WorkstationName"] = nunique(rs, 4624, "workstation")
    f["nuniq_4625_TargetUserName"] = nunique(rs, 4625, "target_user")
    f["active_processes"] = len(pgs)

    # --- B3: behavioral novelty — "have we EVER seen this before?" ---
    # (uses the leak-free, time-ordered first-seen flags from Stage 7)
    f["new_dst_ips"] = sum(1 for r in conns if is_new.get(id(r), {}).get("new_ip"))
    f["new_domains"] = sum(1 for r in rs if r["event_id"] == 22 and is_new.get(id(r), {}).get("new_domain"))
    f["new_images"] = sum(1 for r in rs if r["event_id"] == 1 and is_new.get(id(r), {}).get("new_image"))
    f["new_cmdlines"] = sum(1 for r in rs if r["event_id"] == 1 and is_new.get(id(r), {}).get("new_cmdline"))
    f["new_workstations"] = sum(1 for r in rs if r["event_id"] == 4624 and is_new.get(id(r), {}).get("new_ws"))

    # --- B4: ratios ---
    f["failed_logon_ratio"] = n_4625 / (n_4625 + n_4624) if (n_4625 + n_4624) else 0.0
    f["external_ip_ratio"] = len(ext) / len(conns) if conns else 0.0
    f["inbound_ratio"] = len(inbound) / len(conns) if conns else 0.0

    # --- B5: record-level semantic / TTP signals ---
    f["n_lolbin"] = sum(1 for r in rs if r["event_id"] == 1 and is_lolbin(r.get("image")))
    f["n_lolbin_path_anomaly"] = sum(1 for r in rs if r["event_id"] == 1 and lolbin_path_anomaly(r.get("image")))
    f["n_suspicious_parent_child"] = sum(1 for r in rs if r["event_id"] == 1
                                          and suspicious_parent_child(r.get("parent_image"), r.get("image")))
    f["n_parent_user_mismatch"] = sum(1 for r in rs if r["event_id"] == 1
                                       and parent_user_mismatch(r.get("parent_user"), r.get("user")))
    f["has_encoded_cmd"] = 1 if any(has_encoded(r.get("cmdline")) for r in rs if r["event_id"] == 1) else 0
    f["lsass_access"] = 1 if any(r["event_id"] == 10 and base_name(r.get("target_image")) == "lsass.exe" for r in rs) else 0
    f["runkey_write"] = 1 if any(r["event_id"] in (12, 13) and r.get("target_object")
                                  and "run" in r["target_object"].lower() for r in rs) else 0
    f["sensitive_share"] = 1 if any(r["event_id"] == 5145 and r.get("share_name")
                                     and r["share_name"].lower().rstrip("\\").endswith(("admin$", "c$")) for r in rs) else 0
    f["is_shell_network"] = 1 if any(base_name(r.get("image")) in SHELLS for r in conns) else 0
    f["internal_445_anomaly"] = sum(1 for r in conns if r.get("dst_port") == 445
                                     and not is_external(r.get("dst_ip"))
                                     and base_name(r.get("image")) not in ("explorer.exe", "svchost.exe"))
    f["unusual_port_for_process"] = sum(1 for r in conns if r.get("dst_port")
                                         and is_new.get(id(r), {}).get("new_imgport"))
    f["dns_entropy_max"] = max([shannon_entropy(r.get("query_name")) for r in rs if r["event_id"] == 22], default=0.0)
    f["smb_pipe_activity"] = 1 if (any(r.get("dst_port") == 445 for r in conns) and any(r["event_id"] == 17 for r in rs)) else 0

    # --- B6: presence of rare/high-signal events ---
    for eid in [4698, 4724, 4738, 4771, 24]:
        f[f"has_{eid}"] = 1 if count_id(rs, eid) else 0
    f["has_4776_fail"] = 1 if any(r["event_id"] == 4776 and str(r.get("status")) != "0x0" for r in rs) else 0

    # --- B7: process-derived (aggregated from the global process timeline) ---
    f["n_micro_duration_procs"] = sum(1 for p in pf if p["is_micro_duration"])
    f["n_parent_dead_spawn"] = sum(1 for p in pf if p["parent_dead_before_spawn"])
    f["n_downloads"] = sum(p["downloads"] for p in pf)
    f["max_beacon_regularity"] = max([p["beacon_regularity"] for p in pf if p["beacon_regularity"] is not None], default=0.0)
    f["max_cmd_char_length"] = max([p["cmd_char_length"] for p in pf], default=0)
    f["max_special_char_density"] = max([p["special_char_density"] for p in pf], default=0.0)
    return f

sample_key, sample_rs = max(((k, v) for k, v in buckets.items() if k[0] == real_users[0]),
                             key=lambda kv: len(kv[1]))
sample_vec = window_features(sample_rs)
print(f"\n[STAGE 8] total feature columns per window: {len(sample_vec)}")
print(f"[STAGE 8] busiest window for {sample_key[0]} @ {slot_label(sample_key[1])} "
      f"({len(sample_rs)} events) — non-zero features:")
print({k: v for k, v in sample_vec.items() if v})


[STAGE 8] total feature columns per window: 69
[STAGE 8] busiest window for alice @ 08-09 09:00 (5 events) — non-zero features:
{'n_1': 3, 'n_3': 1, 'n_11': 1, 'nuniq_3_DestinationIp': 1, 'nuniq_3_DestinationPort': 1, 'nuniq_1_Image': 3, 'nuniq_11_TargetFilename': 1, 'active_processes': 3, 'external_ip_ratio': 1.0, 'n_lolbin': 1, 'n_micro_duration_procs': 2, 'max_beacon_regularity': 0.45993827513267826, 'max_cmd_char_length': 62, 'max_special_char_density': 0.24}


## Stage 9 — Zero-filled (user, day, slot) grid

### Fair-comparison check

Each entry in a baseline "hour bucket" is itself a genuine **10-minute window** vector
(the same granularity as the window being scored) — the hour is only used to *group*
windows that share the same time-of-day, e.g. `alice, hour=9` pools together the six
10-minute slots (09:00, 09:10, ... 09:50) across every day in the dataset. It is never
a 60-minute aggregate compared against a 10-minute window. Within a comparison, the 60
pooled values are collapsed into two numbers (median, MAD) and the test window is
compared once against those — not 60 separate pairwise comparisons, and not a plain
mean/stdev (median/MAD resist being dragged around by a few outlier windows).


In [26]:
# =====================================================================
# STAGE 9 — ZERO-FILLED (user, day, 10-min-slot) GRID
# =====================================================================
days = sorted({day_of(slot_of(r["time"])) for r in records})

grid = {}
for u in real_users:
    for d in days:
        day_start = int(datetime.strptime(d, "%Y-%m-%d").replace(tzinfo=timezone.utc).timestamp())
        for i in range(144):
            slot = day_start + i * WINDOW_SEC
            grid[(u, d, slot)] = window_features(buckets.get((u, slot), []))

print(f"\n[STAGE 9] grid cells: {len(grid)} = {len(real_users)} users x {len(days)} days x 144 slots")


[STAGE 9] grid cells: 5760 = 4 users x 10 days x 144 slots


## Stage 10 — Per (user, hour-of-day) Median + MAD baseline

In [27]:
# =====================================================================
# STAGE 10 — PER (user, hour-of-day) MEDIAN + MAD BASELINE
# =====================================================================
def mad(vals):
    m = median(vals)
    d = median([abs(v - m) for v in vals])
    return d if d > 0 else 0.5   # floor to avoid divide-by-zero in z-scores downstream

baseline = defaultdict(lambda: defaultdict(list))
for (u, d, slot), vec in grid.items():
    baseline[u][hour_of(slot)].append(vec)

FEATURE_KEYS = list(sample_vec.keys())   # FIX: use ALL 69 features for reporting,
                                          # not the old prefix-filtered subset that
                                          # silently hid new_*, has_encoded_cmd,
                                          # lsass_access, runkey_write, etc.

def active_count(vec):
    return sum(v for k, v in vec.items() if k.startswith("n_"))

def print_baseline(u=None, h=None):
    us = [u] if u else real_users
    for uname in us:
        hours = [h] if h is not None else sorted(baseline[uname])
        for hh in hours:
            pool = baseline[uname][hh]
            n_act = sum(1 for v in pool if active_count(v) > 0)
            print(f"=== {uname}  hour {hh:02d}:00-{hh:02d}:59  |  "
                  f"{len(pool)} windows | {n_act:2d} active | {len(pool) - n_act:2d} empty ===")
            sig = [(f, median([v[f] for v in pool])) for f in FEATURE_KEYS if any(v[f] for v in pool)]
            for f, m in sig:
                print(f"    {f:30} median={m:6.2f}  mad={mad([v[f] for v in pool]):6.2f}")
            if not sig:
                print("    (no activity in this hour -- all features 0)")
        print()

print(f"\n[STAGE 10] baseline structure: "
      f"{ {u: len(baseline[u]) for u in real_users} } hours-of-day covered per user")
print_baseline("alice", 5)


[STAGE 10] baseline structure: {'alice': 24, 'bob': 24, 'charlie': 24, 'diana': 24} hours-of-day covered per user
=== alice  hour 05:00-05:59  |  60 windows |  0 active | 60 empty ===
    (no activity in this hour -- all features 0)



## Stage 11 — Simulated attack injection + robust z-score anomaly comparison

### Why z-scores alone aren't enough here

With only ~60 windows per (user, hour) and most security-relevant features being rare/binary
(near-always 0), MAD floors at 0.5 and even a clean "never happened before, happened now" event
only scores about 1.35σ — under almost any reasonable threshold. Stage 11 shows the raw z-score
result; Stage 11b shows the gap explicitly; **Stage 11c below is the recommended approach**:
it combines two complementary signals per feature —

- **surprisal** (presence/rarity): "has this ever happened before for this user/hour?"
- **magnitude** (a log-compressed, uncapped version of the robust z-score): "given it DOES
  happen sometimes, is THIS value far outside the normal range?"

`combined = max(surprisal, magnitude)` — not a sum, since summing would double-count when
both fire on the same underlying event. Magnitude is what catches a spike in a feature that
already occurs regularly (e.g. connection count 20x its usual value) — a case pure
presence/rarity would under-score. A synthetic proof of this is included at the end of Stage 11c.

Of the 69 total features, only the ones with a non-zero value in the scored window can
contribute anything at all — the rest are correctly 0 and silently excluded, not hidden.


In [28]:
# =====================================================================
# STAGE 11 — ANOMALY INJECTION + ROBUST Z-SCORE COMPARISON
# =====================================================================
# Simulated intrusion: Office macro spawns encoded PowerShell, which talks
# to an external C2 IP on a non-standard port, touches LSASS, resolves a
# high-entropy DGA-style domain, and is followed by two failed logons from
# an external IP against the same account.
TARGET_USER = "diana"
ANOM_TIME = "2026-08-10T02:00:00Z"   # off-hours for this user (see Stage 3 hour histogram)

anom = [
    dict(time="2026-08-10T02:00:00Z", event_id=1, user=TARGET_USER,
         resolved_user=TARGET_USER, process_guid="{A1}", parent_process_guid="{A0}",
         image="C:\\Users\\Public\\powershell.exe", cmdline="powershell.exe -enc JABzAD0...",
         parent_image="C:\\Program Files\\Microsoft Office\\root\\Office16\\WINWORD.EXE",
         parent_user="CORP\\diana"),
    dict(time="2026-08-10T02:00:05Z", event_id=3, user=TARGET_USER, resolved_user=TARGET_USER,
         process_guid="{A1}", image="C:\\Users\\Public\\powershell.exe",
         dst_ip="203.0.113.99", dst_port=4444, initiated=True),
    dict(time="2026-08-10T02:00:10Z", event_id=3, user=TARGET_USER, resolved_user=TARGET_USER,
         process_guid="{A1}", image="C:\\Users\\Public\\powershell.exe",
         dst_ip="203.0.113.99", dst_port=4444, initiated=True),
    dict(time="2026-08-10T02:00:15Z", event_id=3, user=TARGET_USER, resolved_user=TARGET_USER,
         process_guid="{A1}", image="C:\\Users\\Public\\powershell.exe",
         dst_ip="203.0.113.99", dst_port=4444, initiated=True),
    dict(time="2026-08-10T02:00:20Z", event_id=3, user=TARGET_USER, resolved_user=TARGET_USER,
         process_guid="{A1}", image="C:\\Users\\Public\\powershell.exe",
         dst_ip="203.0.113.99", dst_port=4444, initiated=True),
    dict(time="2026-08-10T02:00:11Z", event_id=10, user=TARGET_USER, resolved_user=TARGET_USER,
         process_guid="{A1}", image="C:\\Users\\Public\\powershell.exe",
         target_image="C:\\Windows\\System32\\lsass.exe"),
    dict(time="2026-08-10T02:00:25Z", event_id=22, user=TARGET_USER, resolved_user=TARGET_USER,
         query_name="x7k2m9q-v4n8.xyz"),
    dict(time="2026-08-10T02:00:40Z", event_id=4625, user=TARGET_USER, resolved_user=TARGET_USER,
         target_user=TARGET_USER, src_ip="203.0.113.7", workstation="WIN-WS04"),
    dict(time="2026-08-10T02:00:45Z", event_id=4625, user=TARGET_USER, resolved_user=TARGET_USER,
         target_user=TARGET_USER, src_ip="203.0.113.7", workstation="WIN-WS04"),
]
# fill every key normalize() would have produced, defaulting missing ones to None,
# so window_features()/downstream code never KeyErrors on a field this synthetic
# event doesn't set.
NORMALIZED_KEYS = list(records[0].keys())
for e in anom:
    for k in NORMALIZED_KEYS:
        e.setdefault(k, None)
    e["is_new_dummy"] = None  # placeholder, not used

# register the injected process in the same global structures real processes use
proc_start["{A1}"] = epoch(anom[0]["time"]); proc_image["{A1}"] = anom[0]["image"]
proc_cmdline["{A1}"] = anom[0]["cmdline"]; proc_parent["{A1}"] = "{A0}"
proc_parent_image["{A1}"] = anom[0]["parent_image"]; proc_parent_user["{A1}"] = "CORP\\diana"
proc_user["{A1}"] = TARGET_USER
conns_by_proc["{A1}"] = [(epoch(e["time"]), e.get("dst_ip"), e.get("dst_port")) for e in anom if e["event_id"] == 3]
proc_feats["{A1}"] = dict(
    unique_ips=1, downloads=0, cmd_char_length=len(anom[0]["cmdline"]),
    special_char_density=special_char_density(anom[0]["cmdline"]),
    is_lolbin=True, lolbin_path_anomaly=True, suspicious_parent_child=True,
    parent_user_mismatch=False, lifetime_seconds=45, is_micro_duration=False,
    parent_dead_before_spawn=False, beacon_regularity=beacon_regularity("{A1}"))

# mark novelty honestly: every one of these IOCs is genuinely first-seen for
# diana (never appears anywhere in the legitimate 10-day baseline)
for e in anom:
    flags = {}
    if e.get("dst_ip"):
        flags["new_ip"] = e["dst_ip"] not in seen_ip[TARGET_USER]
    if e.get("query_name"):
        flags["new_domain"] = e["query_name"] not in seen_domain[TARGET_USER]
    if e["event_id"] == 1 and e.get("image"):
        flags["new_image"] = e["image"] not in seen_image[TARGET_USER]
    if e["event_id"] == 1 and e.get("cmdline"):
        flags["new_cmdline"] = e["cmdline"] not in seen_cmdline[TARGET_USER]
    if e.get("dst_port") and e.get("image"):
        flags["new_imgport"] = (base_name(e["image"]), e["dst_port"]) not in seen_imgport[TARGET_USER]
    is_new[id(e)] = flags

anomaly_vec = window_features(anom)

# FIX (major gap): the original notebook never actually scored the injected
# window against the Median+MAD baseline it built — it only diffed it
# against one arbitrarily-chosen "busiest normal window" for the same user,
# at a DIFFERENT (daytime) hour than the attack. That is not a baseline
# comparison at all. Here we score against the user's OWN hour-matched
# baseline (hour 2, i.e. 02:00-02:59), as the pipeline design requires.
attack_hour = hour_of(slot_of(ANOM_TIME))
hour_pool = baseline[TARGET_USER].get(attack_hour, [])
print(f"\n[STAGE 11] scoring injected attack window for user={TARGET_USER}, "
      f"hour={attack_hour:02d}:00 against {len(hour_pool)} historical windows at that hour")
if hour_pool:
    n_hour_active = sum(1 for v in hour_pool if active_count(v) > 0)
    print(f"[STAGE 11] of those, {n_hour_active} had ANY real activity "
          f"(diana has essentially no legitimate activity at 02:00 in this dataset -- "
          f"see the hour histogram in the report)")

def robust_zscores(vec, pool):
    """Robust z-score per feature: (x - median) / (1.4826 * MAD)."""
    out = {}
    for k in vec:
        vals = [p[k] for p in pool] if pool else [0]
        m = median(vals)
        d = mad(vals)
        z = (vec[k] - m) / (1.4826 * d)
        out[k] = (vec[k], m, d, z)
    return out

zscores = robust_zscores(anomaly_vec, hour_pool)
Z_THRESHOLD = 3.5
flagged = sorted(
    ((k, *v) for k, v in zscores.items() if abs(v[3]) >= Z_THRESHOLD),
    key=lambda t: -abs(t[4]),
)
print(f"\n[STAGE 11] features flagged as anomalous (|z| >= {Z_THRESHOLD}), "
      f"ranked by |z|:")
print(f"  {'feature':30} {'value':>8} {'baseline_med':>13} {'baseline_mad':>13} {'z':>8}")
for k, val, m, d, z in flagged:
    print(f"  {k:30} {val:>8} {m:>13.2f} {d:>13.2f} {z:>8.2f}")

print(f"\n[STAGE 11] {len(flagged)} / {len(anomaly_vec)} features flagged as anomalous "
      f"by robust z-score alone")

# ---------------------------------------------------------------------
# STAGE 11b — supplementary rule for rare / structurally-binary features
# ---------------------------------------------------------------------
# CAVEAT worth surfacing rather than hiding: MAD-based z-scores are a poor
# fit for rare, near-always-zero indicator features (has_encoded_cmd,
# lsass_access, n_suspicious_parent_child, n_lolbin, n_parent_user_mismatch,
# ...). When a feature's entire baseline pool is 0, mad() floors to 0.5,
# so even a single occurrence (0 -> 1) only produces z = 1/(1.4826*0.5)
# ~= 1.35 -- well under a 3.5 threshold, even though "this NEVER happened
# before and just happened" is exactly the kind of signal a security
# analyst cares about. We therefore add a simple, explicit second rule:
# flag any feature whose entire hour-matched baseline pool was zero but
# whose value in the scored window is non-zero.
zero_baseline_flags = sorted(
    (k for k, (val, m, d, z) in zscores.items()
     if val and m == 0 and all(p[k] == 0 for p in hour_pool) and k not in {k2 for k2, *_ in flagged}),
)
print(f"\n[STAGE 11b] additional rare-event indicators that fired despite an "
      f"all-zero baseline (not caught by the z>= {Z_THRESHOLD} rule above "
      f"because of the MAD floor on sparse features):")
for k in zero_baseline_flags:
    print(f"  {k:30} value={anomaly_vec[k]}  (baseline for this user/hour was 0 in all "
          f"{len(hour_pool)} windows)")
if not zero_baseline_flags:
    print("  (none)")

print(f"\n[STAGE 11] TOTAL distinct anomalous features flagged (z-score + rare-event rule): "
      f"{len(flagged) + len(zero_baseline_flags)} / {len(anomaly_vec)}")


[STAGE 11] scoring injected attack window for user=diana, hour=02:00 against 60 historical windows at that hour
[STAGE 11] of those, 0 had ANY real activity (diana has essentially no legitimate activity at 02:00 in this dataset -- see the hour histogram in the report)

[STAGE 11] features flagged as anomalous (|z| >= 3.5), ranked by |z|:
  feature                           value  baseline_med  baseline_mad        z
  max_cmd_char_length                  30          0.00          0.50    40.47
  n_3                                   4          0.00          0.50     5.40
  new_dst_ips                           4          0.00          0.50     5.40
  unusual_port_for_process              4          0.00          0.50     5.40
  dns_entropy_max                   3.875          0.00          0.50     5.23

[STAGE 11] 5 / 69 features flagged as anomalous by robust z-score alone

[STAGE 11b] additional rare-event indicators that fired despite an all-zero baseline (not caught by the z>= 3.5

## Stage 12 — Test any new / suspicious JSONL file against this baseline

### Using this on a new file

`TEST_FILE` should use the **same raw JSONL schema** as your training file (`computer`,
`timestamp`, `event_id`, `User`/`user`, `ProcessGuid`, `Image`, `CommandLine`, `DestinationIp`,
`QueryName`, ... — the same fields `normalize()` already reads). `ingest_test_file()` extends
the *same* process-timeline and per-user "seen so far" structures built during training
(Stages 6-7) with the new file's events before scoring — this matters because without it,
every test-file process/event would incorrectly look like "no process features" / "not novel",
which hides real attacks rather than flagging them. The training baseline itself (Stages 9-10)
is only read here, never modified. A worked example file, `test_events_example.jsonl`, is
provided alongside this notebook.


In [29]:
# =====================================================================
# STAGE 11c — RARITY (SURPRISAL) + SEVERITY-WEIGHTED COMPOSITE RISK SCORE
# =====================================================================
# Why: with only ~60 windows per (user, hour) and most security-relevant
# features being near-always-zero (Bernoulli-like, not Gaussian), MAD
# z-scores systematically under-score genuine "never happened before, just
# happened" events (Stage 11b showed 23 such features that a hard z >= 3.5
# rule missed entirely). This stage replaces the binary flag with a
# continuous, ranked score that scales smoothly with true historical
# rarity AND lets a handful of independently-critical indicators (LSASS
# access, encoded PowerShell, sensitive-share access, suspicious
# parent/child spawns) count heavily even if they were merely "unlikely"
# rather than literally never-seen.
#
#   1) RARITY  (Laplace/add-one-smoothed occurrence probability):
#        p_occur = (count of windows in the hour-matched baseline with
#                    this feature > 0, plus 1) / (N windows in pool, plus 2)
#        surprisal = -log2(p_occur)            [0 = totally normal,
#                                                 grows unbounded as p -> 0]
#      This is well-behaved for sparse binary/count features and does NOT
#      require the Gaussian-ish assumption a z-score makes; a feature that
#      fires in 30/60 historical windows gets a LOW surprisal even though
#      its median is 0, whereas one that has fired 0/60 times gets a HIGH
#      surprisal for the same median -- exactly the distinction z-scores
#      collapse together via the MAD floor.
#
#   2) SEVERITY WEIGHT (fixed, analyst-assigned, independent of statistics):
#      A handful of indicators are inherently high-risk regardless of how
#      "rare" they are for this specific user -- e.g. LSASS access from an
#      unauthorized process is worth investigating the first time it is
#      seen, not only once it is statistically surprising enough.
#
#   3) COMPOSITE SCORE per feature = severity_weight * surprisal, and the
#      WINDOW-LEVEL risk score = sum of the top-N feature contributions.
#      This also naturally supports the "several medium features add up"
#      case (novel domain + novel IP + unusual port, none individually
#      extreme) that a per-feature hard threshold misses.
SEVERITY_WEIGHTS = {
    # critical: near-certain attacker tradecraft if it occurs at all
    "lsass_access": 5.0, "has_encoded_cmd": 4.0, "sensitive_share": 4.0,
    "runkey_write": 4.0, "n_suspicious_parent_child": 3.5,
    "n_parent_user_mismatch": 3.0, "n_lolbin_path_anomaly": 3.0,
    # high: strong indicators, common enough in dual-use tooling to weight
    # slightly below the "critical" tier
    "new_domains": 2.5, "new_dst_ips": 2.0, "unusual_port_for_process": 2.0,
    "n_lolbin": 1.5, "dns_entropy_max": 1.5, "internal_445_anomaly": 1.5,
    "max_beacon_regularity": 1.5, "failed_logon_ratio": 1.5,
    "n_micro_duration_procs": 1.5, "n_parent_dead_spawn": 1.5,
    "new_images": 1.5, "new_cmdlines": 1.5, "new_workstations": 1.0,
    "external_ip_ratio": 1.0, "n_downloads": 1.0,
}
DEFAULT_WEIGHT = 0.5   # everything else (volume counters etc.) still counts, lightly

def surprisal(feature_key, value, pool):
    """-log2 of the Laplace-smoothed probability this feature is ever non-zero
    in the hour-matched baseline. 0 if the observed value itself is 0."""
    if not value:
        return 0.0
    n = len(pool)
    occur = sum(1 for p in pool if p[feature_key])
    p_occur = (occur + 1) / (n + 2)
    return -math.log2(p_occur)

risk_rows = []
for k, val in anomaly_vec.items():
    if not val:
        continue  # feature absent in this window -> contributes nothing either way
    pool_vals = [p[k] for p in hour_pool] if hour_pool else [0]
    m, d = median(pool_vals), mad(pool_vals)

    # --- presence/rarity component: "has this ever happened before?" ---
    s = surprisal(k, val, hour_pool)

    # --- magnitude component: "given it DOES happen, is this value far
    # outside the normal range?" -- catches spikes in features that occur
    # regularly (e.g. n_3 connection count) but at values the pure
    # presence/rarity check would under-score, since "occurs sometimes"
    # already gives it a low surprisal even if THIS value is way outside
    # what "sometimes" normally looks like.
    z = (val - m) / (1.4826 * d)
    magnitude = math.log2(1 + min(abs(z), 50))   # log-compressed, same scale as surprisal

    combined = max(s, magnitude)   # take whichever signal is stronger, not sum
                                    # (sum would double-count when both fire on the
                                    # same underlying event, e.g. a feature that is
                                    # both never-seen AND has an extreme value)
    w = SEVERITY_WEIGHTS.get(k, DEFAULT_WEIGHT)
    risk_rows.append((k, val, w, s, magnitude, combined, w * combined))

risk_rows.sort(key=lambda t: -t[6])
window_risk_score = sum(t[6] for t in risk_rows)

print(f"\n[STAGE 11c] rarity + magnitude + severity composite score for the injected window "
      f"(user={TARGET_USER}, hour={attack_hour:02d}:00):")
print(f"  {'feature':28} {'value':>7} {'weight':>6} {'surprisal':>9} {'magnitude':>9} {'combined':>8} {'contrib':>9}")
for k, val, w, s, mag, comb, contrib in risk_rows:
    print(f"  {k:28} {val:>7} {w:>6.1f} {s:>9.2f} {mag:>9.2f} {comb:>8.2f} {contrib:>9.2f}")
print(f"  {'-'*28} {'-'*7} {'-'*6} {'-'*9} {'-'*9} {'-'*8} {'-'*9}")
print(f"  {'TOTAL':28} {'':>7} {'':>6} {'':>9} {'':>9} {'':>8} {sum(t[6] for t in risk_rows):>9.2f}")
print(f"\n[STAGE 11c] WINDOW RISK SCORE (sum of ALL {len(risk_rows)} contributing "
      f"features out of {len(anomaly_vec)} total -- the other "
      f"{len(anomaly_vec) - len(risk_rows)} were 0 in this window and contribute nothing): "
      f"{window_risk_score:.2f}")
print("[STAGE 11c] note: for this specific demo, diana's hour-02 baseline is entirely")
print("  zero for every feature, so magnitude and surprisal happen to coincide (both")
print("  reduce to the same 'never seen before' signal). magnitude only diverges from")
print("  surprisal -- and adds real value -- for features that DO occur sometimes in a")
print("  user's normal baseline but spike far outside their usual range; see the")
print("  synthetic example directly below.")

# ---------------------------------------------------------------------
# Minimal synthetic proof that magnitude adds information surprisal alone
# would miss: a feature that occurs in HALF the baseline windows (so
# presence/rarity treats it as unremarkable) but spikes to 20x its normal
# value in the scored window.
demo_pool_vals = [2, 3, 0, 4, 2, 0, 3, 0, 2, 0] * 6     # occurs in 6/10 -> "not rare"
demo_value = 60                                          # a 20-30x spike
demo_pool = [{"demo": v} for v in demo_pool_vals]
demo_s = surprisal("demo", demo_value, demo_pool)
demo_m, demo_d = median(demo_pool_vals), mad(demo_pool_vals)
demo_z = (demo_value - demo_m) / (1.4826 * demo_d)
demo_mag = math.log2(1 + min(abs(demo_z), 50))
print(f"\n[STAGE 11c demo] synthetic feature that occurs in {sum(1 for v in demo_pool_vals if v)}/"
      f"{len(demo_pool_vals)} baseline windows (NOT rare), but spikes to {demo_value} "
      f"(baseline median={demo_m}):")
print(f"    surprisal-only score : {demo_s:.2f}  (looks unremarkable -- it happens often)")
print(f"    magnitude score      : {demo_mag:.2f}  (correctly catches the extreme spike)")
print(f"    combined (max) score : {max(demo_s, demo_mag):.2f}")
print("[STAGE 11c] compare against a normal (non-attack) window's risk score for scale:")

def window_risk(vec, pool):
    """Same combined surprisal+magnitude+severity scoring used above, factored
    out so the 'normal' comparison window is scored with the IDENTICAL formula
    (not a shortcut version) -- otherwise the two numbers wouldn't be comparable."""
    total = 0.0
    for k, val in vec.items():
        if not val:
            continue
        pool_vals = [p[k] for p in pool] if pool else [0]
        m, d = median(pool_vals), mad(pool_vals)
        s = surprisal(k, val, pool)
        z = (val - m) / (1.4826 * d)
        magnitude = math.log2(1 + min(abs(z), 50))
        combined = max(s, magnitude)
        total += SEVERITY_WEIGHTS.get(k, DEFAULT_WEIGHT) * combined
    return total

normal_key, normal_rs = max(((k, v) for k, v in buckets.items() if k[0] == TARGET_USER),
                             key=lambda kv: len(kv[1]))
normal_vec = window_features(normal_rs)
normal_hour_pool = baseline[TARGET_USER].get(hour_of(normal_key[1]), [])
normal_risk = window_risk(normal_vec, normal_hour_pool)
print(f"  normal busiest window ({slot_label(normal_key[1])}) risk score: {normal_risk:.2f}")
print(f"  injected attack window risk score                 : {window_risk_score:.2f}")


[STAGE 11c] rarity + magnitude + severity composite score for the injected window (user=diana, hour=02:00):
  feature                        value weight surprisal magnitude combined   contrib
  lsass_access                       1    5.0      5.95      1.23     5.95     29.77
  has_encoded_cmd                    1    4.0      5.95      1.23     5.95     23.82
  n_suspicious_parent_child          1    3.5      5.95      1.23     5.95     20.84
  n_lolbin_path_anomaly              1    3.0      5.95      1.23     5.95     17.86
  new_domains                        1    2.5      5.95      1.23     5.95     14.89
  new_dst_ips                        4    2.0      5.95      2.68     5.95     11.91
  unusual_port_for_process           4    2.0      5.95      2.68     5.95     11.91
  new_images                         1    1.5      5.95      1.23     5.95      8.93
  new_cmdlines                       1    1.5      5.95      1.23     5.95      8.93
  failed_logon_ratio               1.0   

## Stage 13

In [ ]:
# =====================================================================
# STAGE 12 — TEST ANY NEW / SUSPICIOUS JSONL FILE AGAINST THIS BASELINE
# =====================================================================
# Point TEST_FILE at a JSONL file using the SAME RAW SCHEMA as your
# training data. It is run through the identical filtering -> normalization
# -> correlation -> novelty -> feature steps, then scored against the
# Median+MAD baseline already built above. The training baseline itself is
# NOT rebuilt or modified -- only read.
#
# window_features() looks up process behavior in proc_feats and novelty in
# is_new, BOTH populated only for the training file. ingest_test_file()
# below extends the SAME process-timeline and per-user "seen so far"
# structures with the new file's events before scoring, so a test file is
# judged fairly against everything known about that user up to now.

def ingest_test_file(path):
    """Load + normalize a new JSONL file and extend the process-timeline and
    per-user novelty tracking with its events. Does NOT touch the training
    grid/baseline."""
    test_raw = [json.loads(line) for line in open(path, encoding="utf-8")]
    test_kept = [r for r in test_raw if r.get("event_id") in KEEP_EVENTS]
    dropped_n = len(test_raw) - len(test_kept)
    test_norm = [normalize(r) for r in test_kept]

    for r in test_norm:
        r["resolved_user"], r["resolved_sid"] = resolve_user(r)
        sess = session_of(r)
        if sess:
            r["login_ip"] = sess["src_ip"]; r["login_type"] = sess["logon_type"]
            r["login_workstation"] = sess["workstation"]
        else:
            r["login_ip"] = r["login_type"] = r["login_workstation"] = None

    # --- extend the global process timeline (mirrors Stage 6) ---
    for r in test_norm:
        pg = r.get("process_guid")
        if not pg:
            continue
        if r["event_id"] == 1:
            proc_start[pg] = epoch(r["time"]); proc_image[pg] = r.get("image")
            proc_cmdline[pg] = r.get("cmdline"); proc_parent[pg] = r.get("parent_process_guid")
            proc_parent_image[pg] = r.get("parent_image"); proc_parent_user[pg] = r.get("parent_user")
            proc_user[pg] = r.get("user")
        elif r["event_id"] == 5:
            proc_end[pg] = epoch(r["time"])
        elif r["event_id"] == 3:
            conns_by_proc[pg].append((epoch(r["time"]), r.get("dst_ip"), r.get("dst_port")))
        elif r["event_id"] == 15:
            downloads_by_proc[pg] += 1

    new_pgs = {r.get("process_guid") for r in test_norm if r.get("process_guid")}
    for pg in new_pgs - set(proc_feats):
        if pg not in proc_start:
            continue
        lifetime = proc_end.get(pg, proc_start[pg]) - proc_start[pg]
        img = proc_image.get(pg) or ""
        cmdl = proc_cmdline.get(pg) or ""
        parent_pg = proc_parent.get(pg)
        proc_feats[pg] = {
            "unique_ips": len({c[1] for c in conns_by_proc[pg]}),
            "downloads": downloads_by_proc.get(pg, 0),
            "cmd_char_length": len(cmdl),
            "special_char_density": special_char_density(cmdl),
            "is_lolbin": is_lolbin(img),
            "lolbin_path_anomaly": lolbin_path_anomaly(img),
            "suspicious_parent_child": suspicious_parent_child(proc_parent_image.get(pg), img),
            "parent_user_mismatch": parent_user_mismatch(proc_parent_user.get(pg), proc_user.get(pg)),
            "lifetime_seconds": lifetime,
            "is_micro_duration": lifetime < 5,
            "parent_dead_before_spawn": bool(parent_pg and parent_pg in proc_end
                                              and proc_end[parent_pg] < proc_start[pg]),
            "beacon_regularity": beacon_regularity(pg),
        }

    # --- extend per-user novelty tracking, in time order (mirrors Stage 7) ---
    by_user = defaultdict(list)
    for r in test_norm:
        if r["resolved_user"]:
            by_user[r["resolved_user"]].append(r)
    for u, urecs in by_user.items():
        for r in sorted(urecs, key=lambda r: r["time"]):
            flags = {}
            if r.get("dst_ip"):
                flags["new_ip"] = r["dst_ip"] not in seen_ip[u]; seen_ip[u].add(r["dst_ip"])
            if r.get("query_name"):
                flags["new_domain"] = r["query_name"] not in seen_domain[u]; seen_domain[u].add(r["query_name"])
            if r["event_id"] == 1 and r.get("image"):
                flags["new_image"] = r["image"] not in seen_image[u]; seen_image[u].add(r["image"])
            if r["event_id"] == 1 and r.get("cmdline"):
                flags["new_cmdline"] = r["cmdline"] not in seen_cmdline[u]; seen_cmdline[u].add(r["cmdline"])
            if r["event_id"] == 4624 and r.get("workstation"):
                flags["new_ws"] = r["workstation"] not in seen_ws[u]; seen_ws[u].add(r["workstation"])
            if r.get("dst_port") and r.get("image"):
                key = (base_name(r["image"]), r["dst_port"])
                flags["new_imgport"] = key not in seen_imgport[u]; seen_imgport[u].add(key)
            is_new[id(r)] = flags

    print(f"[TEST] {path}: loaded {len(test_raw)} raw records, kept {len(test_kept)} "
          f"after event filtering ({dropped_n} dropped -- unrecognized event_id)")
    return test_norm

def window_risk_detail(vec, pool):
    """Composite scoring (surprisal + magnitude + severity) that ALSO returns
    the per-feature breakdown so a scored window can show WHY it scored."""
    rows = []
    total = 0.0
    for k, val in vec.items():
        if not val:
            continue
        vals = [p[k] for p in pool] if pool else [0]
        m, d = median(vals), mad(vals)
        s = surprisal(k, val, pool)                       # rarity component
        z = (val - m) / (1.4826 * d)                      # magnitude component
        magnitude = math.log2(1 + min(abs(z), 50))
        combined = s + magnitude                          # both count
        w = SEVERITY_WEIGHTS.get(k, DEFAULT_WEIGHT)
        contrib = w * combined
        rows.append((k, val, w, s, magnitude, combined, contrib))
        total += contrib
    rows.sort(key=lambda t: -t[6])                        # rank by contribution
    return total, rows

def score_test_file(path):
    """Ingest a new JSONL file, bucket it into 10-minute user windows, and
    score each window against the user's existing hour-matched baseline,
    showing the per-feature surprisal / magnitude / combined / contribution."""
    test_norm = ingest_test_file(path)
    test_buckets = defaultdict(list)
    for r in test_norm:
        test_buckets[(r["resolved_user"] or "UNATTRIBUTED", slot_of(r["time"]))].append(r)

    results = []
    for (u, slot), rs in sorted(test_buckets.items(), key=lambda kv: kv[0][1]):
        label = slot_label(slot)
        if u == "UNATTRIBUTED":
            print(f"\n[TEST] {label}: {len(rs)} event(s) could not be attributed to a known "
                  f"user (SYSTEM/service activity) -- skipped, no per-user baseline applies.")
            continue
        vec = window_features(rs)
        if u not in baseline:
            print(f"\n[TEST] {label}, user={u}: NO existing baseline for this user "
                  f"(never appeared in the training data) -- showing raw features, not scored:")
            print("   ", {k: v for k, v in vec.items() if v})
            continue
        hpool = baseline[u].get(hour_of(slot), [])
        if not hpool:
            print(f"\n[TEST] {label}, user={u}, hour={hour_of(slot):02d}:00: no historical "
                  f"windows for this user at this hour -- scored as if that hour has always "
                  f"been empty for them (a real off-hours flag, not a data gap in this file).")

        score, rows = window_risk_detail(vec, hpool)
        results.append((slot, u, score))
        n_nonzero = len(rows)
        print(f"\n[TEST] {label}  user={u:10} risk_score={score:8.2f}  "
              f"({n_nonzero} / {len(vec)} nonzero features)")
        print(f"    {'feature':28} {'value':>8} {'wt':>5} "
              f"{'surprisal':>9} {'magnitude':>9} {'combined':>8} {'contrib':>8}")
        for k, val, w, s, mag, comb, contrib in rows:
            print(f"    {k:28} {val:>8} {w:>5.1f} {s:>9.2f} "
                  f"{mag:>9.2f} {comb:>8.2f} {contrib:>8.2f}")

    if len(results) > 1:
        print("\n[TEST] summary across all windows in this file, ranked by risk score:")
        for slot, u, score in sorted(results, key=lambda t: -t[2]):
            print(f"    {slot_label(slot):16} user={u:10} risk_score={score:8.2f}")
    return results

print("[STAGE 12] score_test_file(path) is ready to use -- see the input cell below.")

# =====================================================================
# INPUT CELL — set TEST_FILE and run
# =====================================================================
TEST_FILE = "test_normal_activity.jsonl"   # <-- change this to your file's path

if os.path.exists(TEST_FILE):
    score_test_file(TEST_FILE)
else:
    print(f"[TEST] '{TEST_FILE}' not found in the working directory.")
    print("       Set TEST_FILE above to the path of your JSONL log file and re-run this cell.")

[STAGE 12] score_test_file(path) is ready to use -- see the input cell below.
[TEST] normal.jsonl: loaded 1096 raw records, kept 1096 after event filtering (0 dropped -- unrecognized event_id)

[TEST] 08-01 08:50  user=alice      risk_score=   23.86  (9 / 69 nonzero features)
    feature                         value    wt surprisal magnitude combined  contrib
    n_micro_duration_procs              1   1.5      2.95      1.23     4.19     6.28
    max_cmd_char_length                25   0.5      2.95      5.12     8.07     4.04
    n_1                                 1   0.5      2.95      1.23     4.19     2.09
    nuniq_1_Image                       1   0.5      2.95      1.23     4.19     2.09
    active_processes                    1   0.5      2.95      1.23     4.19     2.09
    n_4624                              1   0.5      2.49      1.23     3.73     1.86
    n_4768                              1   0.5      2.49      1.23     3.73     1.86
    nuniq_4624_WorkstationName     